## Паплайн для загрузки данных, предобработки, генерации прогнозов и расчета метрик

Здесь считаем метрики по массовой загрузке классификации универсальных категорий за период с даты фиксации тренировочного датасета - 09.10.2025 по 03.12.2025. Классификация делалась для платежей, по которым не было меток (несколько дней были ранее классифицированы с помощью yandexgpt) и по которым доступны все вспомогательные поля (статья, проект, тип контаргента). Были загружен порядка 35 тысяч меток.
Здесь считаем только по прогнозам соответствующим ранее составленному словарю для разметки тренировочного датасета "статья - универсальная категория".

In [2]:
import pandas as pd
import numpy as np
import random
import os
import re
import requests

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import IncrementalPCA

# ⏳ прогресс-бары
import tqdm as tqdm_lib
from tqdm import tqdm

# 🧠 обработка текста и NLP
import spacy
try:
    import transformers
    from transformers import AutoModel, AutoTokenizer
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers"], stdout=subprocess.DEVNULL)
    import transformers
    from transformers import AutoModel, AutoTokenizer    
    
# 🤖 pyTorch
import torch

# загрузка моделей и функци для предобработки данных
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report,roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import label_binarize

from imblearn.combine import SMOTEENN
from imblearn.under_sampling import RandomUnderSampler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import joblib

import catboost
from catboost import CatBoostClassifier, utils


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 0)
pd.set_option('display.max_colwidth', 120)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [8]:
slovar = pd.read_csv('../../uc_data_labeling/articles_uc_added.csv')

In [ ]:
data_full_prepared_post_mass_insert = pd.read_parquet('data_full_prepared_post_mass_insert.parquet')

data_download_after_mass_insert = pd.read_parquet('data_download_after_mass_insert.parquet')

In [6]:
data = data_download_after_mass_insert[data_download_after_mass_insert.id.isin(data_full_prepared_post_mass_insert.id)].copy()
display(data.id.count())

(data.uc__uc_id.values == 
 data_full_prepared_post_mass_insert.uc__uc_id.values).sum()

35349

35349

In [9]:
data['articles__name'] = data['articles__name'].str.lower()

data = data.merge(
    slovar[['raw', 'uc_id']],
    left_on='articles__name',
    right_on='raw',
    how='left'
)

data.rename(columns={'uc_id': 'uc_id_true'}, inplace=True)
data.drop(columns=['raw'], inplace=True)
data.head(3)

,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,uc__uc_id,uc_id_true
0,6161,26,51,2022-03-31,100.00,ДОБРОВОЛЬНОЕ ПОЖЕРТВОВАНИЕ;Сумма 100.00 руб.;Комиссия 0.00 руб;Дата оплаты 31/03/2022;,0,incoming,41,0,73,0,0,26.0,43.0,NaN,NaN,NaN,None,41.0,43.0,0.0,Уставные цели,NaN,NaN,NaN,None,NaN,NaN,NaN,1.0,NaN
1,6661,30,1226,2022-11-30,300.00,None,0,incoming,0,0,0,0,218,30.0,42.0,NaN,NaN,NaN,None,NaN,NaN,NaN,None,218.0,42.0,216.0,Массовые разовые,NaN,NaN,NaN,5.0,NaN
2,7694,38,1061,2023-01-24,9826.99,Оплата по заявлению на возврат средств №б/н#30 от 17.01.23 (в т.ч. НДС 1637.83 руб.).,0,incoming,97,272,0,0,0,38.0,45.0,NaN,NaN,NaN,None,97.0,45.0,53.0,Программа_Поддержка семьи,NaN,NaN,NaN,None,NaN,NaN,NaN,6.0,NaN


In [10]:
data_new = data[data.uc_id_true.isna()]
data_new.id.count()

3166

In [ ]:
# сбрасываем строки в которые не смогли проставить истинные метки из-за статей ранее не включенных в словарь разметки
# эти строки по идее показывают обобщащую способность модели на новых, ранее не виденных данных
data.dropna(subset=['uc_id_true'], inplace=True)
data.id.count()

32183

In [12]:
# метрики прогнозов cb
accuracy = accuracy_score(data['uc_id_true'], data['uc__uc_id'])
precision = precision_score(data['uc_id_true'], data['uc__uc_id'], average='weighted')
recall = recall_score(data['uc_id_true'], data['uc__uc_id'], average='macro')
f1 = f1_score(data['uc_id_true'], data['uc__uc_id'], average='weighted')


print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)

print("\nОтчет по классам:")
print(classification_report(data['uc_id_true'], data['uc__uc_id']))

Accuracy: 0.9657893919149861
Precision (weighted): 0.9715178786412009
Recall (macro): 0.9092292277840903
F1-score (weighted): 0.9676235393471254

Отчет по классам:
              precision    recall  f1-score   support

         1.0       1.00      0.97      0.98     20893
         2.0       0.98      0.97      0.97      8825
         3.0       0.60      0.84      0.70       343
         4.0       0.94      0.82      0.88       628
         5.0       0.73      0.87      0.79       540
         6.0       0.74      0.89      0.81       590
         7.0       0.61      0.99      0.75       245
         8.0       0.73      0.99      0.84        82
         9.0       0.56      0.84      0.67        37

    accuracy                           0.97     32183
   macro avg       0.76      0.91      0.82     32183
weighted avg       0.97      0.97      0.97     32183



In [13]:
data_new.id.count()

3166

In [ ]:
data_new.articles__name.nunique()

54

In [31]:
data_new.articles__name.unique()

array([None, 'продажи через магазины', 'сбер', 'колесников а.а.',
       'подопечный: маргамов аскар', 'выручка от консультаций',
       'озон "забота"_ип', 'мкб_терминал', 'коллаборации',
       'миксплат_рекурренты', 'пожертвование от томшинский',
       'физ лица через озон (интернет решения)',
       'сбербанк приложение мерчант №211000044429',
       'озон "забота" сертификаты', 'мтс', 'фонды и нко', 'тпэй',
       'пожертвование от бф сделай', 'товары для нко.яндекс',
       'вк добро физлица', 'т-банк физлица', 'дпд зоогостиница',
       'туба физлица', '3. сбп озон банк', '1. сайт qr',
       'дпд маркетплейс озон', 'пожертвования юр лица',
       'массовые (vk donut)', 'совкомбанк эквайринг - найти?',
       'втб физлица', '4. тбанк благотворительность', '2. сайт_клауд',
       'дпд депозит альфа', 'подопечный: нечаева мирослава',
       'активный гражданин платформа', 'помощь рядом_платформа',
       'подопечный: галиев арслан', 'юмани нэк.134850.06',
       'пожертвование че

In [87]:
data_new.projects__name.isna().sum()

84

In [17]:
data_download_after_mass_insert.articles__name.nunique()

808

In [18]:
proverka = set(data_download_after_mass_insert.articles__name.str.lower().unique())-set(slovar.raw.str.lower().unique())
len(proverka)

127

In [24]:
proverka

{'02.3. гранты',
 '1. сайт qr',
 '2. сайт_клауд',
 '3. сбп озон банк',
 '4. тбанк благотворительность',
 '5. озон "забота"',
 None,
 '_родитель: валеева вероника вилевна',
 '_родитель: губайдуллин рустам энгелевич',
 '_родитель: камалова екатерина павловна',
 '_родитель: корнеев сергей николаевич',
 '_родитель: лукманов ильдар рамилевич',
 '_родитель: мубаширова регина линировна',
 '_родитель: новикова вероника борисовна',
 '_родитель: попова виктория викторовна',
 '_родитель: сухарева дина равиловна',
 '_родитель: хазиев данил радмирович',
 '_родитель: яппаров айдар айратович',
 'ozon банк. удвоение',
 'активный гражданин платформа',
 'взносы учредителя',
 'вк добро физлица',
 'внесение на р/сч налички',
 'втб физлица',
 'выручка от консультаций',
 'глобус',
 'грант мэра 2025',
 'грант от абсолют',
 'гранты мэра архив',
 'дпд глэмпинг',
 'дпд депозит альфа',
 'дпд депозит сбер',
 'дпд зоогостиница',
 'дпд маркетплейс озон',
 'дпд маркетплейс ямаркет',
 'дпд партнерство',
 'зао "атлант